Step 2

In [ ]:
"""
Espoo District Heating Network Optimization
"""


import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
import dhnx
import os
import pandas as pd


import os

# Adjust this path to match your actual project structure
base_dir = r"C:\Users\PC\Documents\KTH Master 2\Practical Optimization of Energy Networks\Network_optmisation\Project__DH"
twn_data_path = os.path.join(base_dir, "DHNx_files", "Espoo", "twn_data_step2")
invest_data_path = os.path.join(base_dir, "DHNx_files", "Espoo", "invest_data_step2")

# REMOVE .DS_Store file if it exists (macOS issue)
ds_store_path = os.path.join(invest_data_path, ".DS_Store")
if os.path.exists(ds_store_path):
    os.remove(ds_store_path)
    print(f"  - Removed .DS_Store file")


print('='*60)
print('ESPOO DHN - INVESTMENT OPTIMIZATION')
print('='*60)

# Load network
print('\n[1/3] Loading network...')
network = dhnx.network.ThermalNetwork()
network = network.from_csv_folder(twn_data_path)
invest_opt = dhnx.input_output.load_invest_options(invest_data_path)

print(f'  - Producers: {len(network.components.producers)}')
print(f'  - Consumers: {len(network.components.consumers)}')
print(f'  - Pipe segments: {len(network.components.pipes)}')

# Plot initial network topology - ALL PIPES IN GRAY
print('\nPlotting initial network (all pipes in gray)...')

# Create figure with equal dimensions
fig, ax = plt.subplots(figsize=(10, 10))

# Create a mapping of all node IDs to coordinates
node_coords = {}

# Add all nodes with their coordinates
for idx, row in network.components.producers.iterrows():
    node_coords[f'producers-{idx}'] = (row['lon'], row['lat'])

for idx, row in network.components.consumers.iterrows():
    node_coords[f'consumers-{idx}'] = (row['lon'], row['lat'])

for idx, row in network.components.forks.iterrows():
    node_coords[f'forks-{idx}'] = (row['lon'], row['lat'])

# Draw ALL pipes in light gray
for idx, pipe in network.components.pipes.iterrows():
    from_node = pipe['from_node']
    to_node = pipe['to_node']
    
    # Get coordinates
    if from_node in node_coords and to_node in node_coords:
        x_coords = [node_coords[from_node][0], node_coords[to_node][0]]
        y_coords = [node_coords[from_node][1], node_coords[to_node][1]]
        
        ax.plot(x_coords, y_coords, color='lightgray', linewidth=2, alpha=0.7, zorder=1)

# Add nodes on top
ax.scatter(network.components.consumers['lon'], network.components.consumers['lat'],
          color='tab:green', s=100, edgecolors='black', linewidths=1, zorder=3, label='Consumers')
ax.scatter(network.components.producers['lon'], network.components.producers['lat'],
          color='tab:red', s=150, edgecolors='black', linewidths=1, zorder=3, label='Producers')
ax.scatter(network.components.forks['lon'], network.components.forks['lat'],
          color='tab:grey', s=50, edgecolors='black', linewidths=0.5, zorder=2, label='Forks')

ax.set_title('Espoo DHN - Initial Network Topology', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(24.14, 24.20)
plt.tight_layout()
plt.savefig('Outputs/network_initial_step2.1.png', dpi=150, bbox_inches='tight')
plt.close()


# Run optimization with MIPGAP for faster solving
print('\n[2/3] Running optimization with MIPGAP=0.05 (5% optimality gap)...')
# GLPK mipgap must be nested in options dict within solve_kw
network.optimize_investment(invest_options=invest_opt, solver='gurobi', 
                           solve_kw={'tee': True, 'options': {'mipgap': 0.001}})


# Get results
results = network.results.optimization['components']['pipes']
results.to_csv("Outputs/optimization_results.csv")

# Summary
print('\n[3/3] Results:')
print('-'*60)
objective = network.results.optimization['oemof_meta']['objective']
print(f'Total Cost: {objective:,.0f} EUR')

# Pipe types used
active_pipes = results[results['capacity'] > 0.001]
pipe_counts = active_pipes['hp_type'].value_counts()
print(f'\nPipes installed:')
for pipe_type, count in pipe_counts.items():
    total_cap = active_pipes[active_pipes['hp_type'] == pipe_type]['capacity'].sum()
    print(f'  {pipe_type}: {count} segments ({total_cap:.1f} kW)')

# Plot optimized network with pipe types color-coded - KEEP EXACTLY AS BEFORE
print('\nCreating optimized network plot with pipe types...')

# Define colors for different pipe types
pipe_colors = {
    'DN25': 'yellow',
    'DN30': 'orange', 
    'DN40': 'red',
    'DN50': 'purple',
    'DN60': 'blue'
}

# Create figure with equal dimensions
fig, ax = plt.subplots(figsize=(10, 10))

# Create a mapping of all node IDs to coordinates (recreate for consistency)
node_coords = {}

# Add all nodes with their coordinates
for idx, row in network.components.producers.iterrows():
    node_coords[f'producers-{idx}'] = (row['lon'], row['lat'])

for idx, row in network.components.consumers.iterrows():
    node_coords[f'consumers-{idx}'] = (row['lon'], row['lat'])

for idx, row in network.components.forks.iterrows():
    node_coords[f'forks-{idx}'] = (row['lon'], row['lat'])

# Draw pipes grouped by type - EXACTLY AS BEFORE
from matplotlib.lines import Line2D

for pipe_type in sorted(active_pipes['hp_type'].unique()):
    pipes_of_type = active_pipes[active_pipes['hp_type'] == pipe_type]
    color = pipe_colors.get(pipe_type, 'black')
    
    for idx, pipe in pipes_of_type.iterrows():
        from_node = pipe['from_node']
        to_node = pipe['to_node']
        
        # Get coordinates
        if from_node in node_coords and to_node in node_coords:
            x_coords = [node_coords[from_node][0], node_coords[to_node][0]]
            y_coords = [node_coords[from_node][1], node_coords[to_node][1]]
            
            ax.plot(x_coords, y_coords, color=color, linewidth=3, alpha=0.8, zorder=1)

# Add nodes on top
ax.scatter(network.components.consumers['lon'], network.components.consumers['lat'],
          color='tab:green', s=100, edgecolors='black', linewidths=1, zorder=3, label='Consumers')
ax.scatter(network.components.producers['lon'], network.components.producers['lat'],
          color='tab:red', s=150, edgecolors='black', linewidths=1, zorder=3, label='Producers')
ax.scatter(network.components.forks['lon'], network.components.forks['lat'],
          color='tab:grey', s=50, edgecolors='black', linewidths=0.5, zorder=2, label='Forks')

# Create custom legend for pipe types
pipe_legend_elements = [Line2D([0], [0], color=pipe_colors.get(pt, 'black'), linewidth=3, 
                               label=f'{pt} ({pipe_counts[pt]} seg)')
                       for pt in sorted(pipe_counts.index)]

# Get handles and labels from existing legend
handles, labels = ax.get_legend_handles_labels()

# Combine pipe types and node types in legend
all_handles = pipe_legend_elements + handles
ax.legend(handles=all_handles, loc='best', fontsize=9)

ax.set_title('Espoo DHN - Optimized Network with Pipe Types', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(24.14, 24.20)
plt.tight_layout()
plt.savefig('Outputs/network_optimized_step2.1.png', dpi=150, bbox_inches='tight')
plt.close()


ESPOO DHN - INVESTMENT OPTIMIZATION

[1/3] Loading network...
  - Producers: 5
  - Consumers: 10
  - Pipe segments: 20

Plotting initial network (all pipes in gray)...

[2/3] Running optimization with MIPGAP=0.05 (5% optimality gap)...


c:\Users\PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\oemof\solph\flows\_flow.py:163: FutureWarning: For backward compatibility, the option investment overwrites the option nominal_value. Both options cannot be set at the same time.
  warn(msg, FutureWarning)
c:\Users\PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\oemof\network\network\nodes.py:250: FutureWarning: Usage of oemof.network.Component is deprecated. Use oemof.network.Node instead.
  warnings.warn(


Set parameter Username
Academic license - for non-commercial use only - expires 2026-12-11
Read LP format model from file C:\Users\PC\AppData\Local\Temp\tmp3kzaqjls.pyomo.lp
Reading time = 0.07 seconds
x1: 1146 rows, 1005 columns, 2730 nonzeros
Set parameter MIPGap to value 0.001
Gurobi Optimizer version 11.0.1 build v11.0.1rc0 (win64 - Windows 11+.0 (26100.2))

CPU model: Intel(R) Core(TM) i5-8350U CPU @ 1.70GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 1146 rows, 1005 columns and 2730 nonzeros
Model fingerprint: 0x1f6d38f0
Variable types: 880 continuous, 125 integer (125 binary)
Coefficient statistics:
  Matrix range     [7e-03, 8e+03]
  Objective range  [3e+03, 2e+06]
  Bounds range     [1e+00, 8e+03]
  RHS range        [7e+01, 1e+03]
Presolve removed 785 rows and 635 columns
Presolve time: 0.12s
Presolved: 361 rows, 370 columns, 1185 nonzeros
Variable types: 245 continuous, 125 integer (125 bi

INFO:root:Optimization successful...



[3/3] Results:
------------------------------------------------------------
Total Cost: 105,396,784 EUR

Pipes installed:
  DN25: 7 segments (1670.7 kW)
  DN30: 6 segments (4180.5 kW)
  DN40: 2 segments (2273.8 kW)
  DN50: 1 segments (2360.8 kW)

Creating optimized network plot with pipe types...


In [2]:
# Define colors for different pipe types
pipe_colors = {
    'DN25': 'yellow',
    'DN30': 'orange', 
    'DN40': 'red',
    'DN50': 'purple',
    'DN60': 'blue'
}

# Create figure with equal dimensions
fig, ax = plt.subplots(figsize=(10, 10))

# Create a mapping of all node IDs to coordinates (recreate for consistency)
node_coords = {}

# Add all nodes with their coordinates
for idx, row in network.components.producers.iterrows():
    node_coords[f'producers-{idx}'] = (row['lon'], row['lat'])

for idx, row in network.components.consumers.iterrows():
    node_coords[f'consumers-{idx}'] = (row['lon'], row['lat'])

for idx, row in network.components.forks.iterrows():
    node_coords[f'forks-{idx}'] = (row['lon'], row['lat'])

# Draw pipes grouped by type - EXACTLY AS BEFORE
from matplotlib.lines import Line2D

for pipe_type in sorted(active_pipes['hp_type'].unique()):
    pipes_of_type = active_pipes[active_pipes['hp_type'] == pipe_type]
    color = pipe_colors.get(pipe_type, 'black')
    
    for idx, pipe in pipes_of_type.iterrows():
        from_node = pipe['from_node']
        to_node = pipe['to_node']
        
        # Get coordinates
        if from_node in node_coords and to_node in node_coords:
            x_coords = [node_coords[from_node][0], node_coords[to_node][0]]
            y_coords = [node_coords[from_node][1], node_coords[to_node][1]]
            
            ax.plot(x_coords, y_coords, color=color, linewidth=3, alpha=0.8, zorder=1)

# Add nodes on top
ax.scatter(network.components.consumers['lon'], network.components.consumers['lat'],
          color='tab:green', s=100, edgecolors='black', linewidths=1, zorder=3, label='Consumers')
ax.scatter(network.components.producers['lon'], network.components.producers['lat'],
          color='tab:red', s=150, edgecolors='black', linewidths=1, zorder=3, label='Producers')
ax.scatter(network.components.forks['lon'], network.components.forks['lat'],
          color='tab:grey', s=50, edgecolors='black', linewidths=0.5, zorder=2, label='Forks')

# Create custom legend for pipe types
pipe_legend_elements = [Line2D([0], [0], color=pipe_colors.get(pt, 'black'), linewidth=3, 
                               label=f'{pt} ({pipe_counts[pt]} seg)')
                       for pt in sorted(pipe_counts.index)]

# Get handles and labels from existing legend
handles, labels = ax.get_legend_handles_labels()

# Combine pipe types and node types in legend
all_handles = pipe_legend_elements + handles
ax.legend(handles=all_handles, loc='best', fontsize=9)

ax.set_title('Espoo DHN - Optimized Network with Pipe Types', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(24.14, 24.20)
plt.tight_layout()
plt.savefig('Outputs/network_optimized_step2.gurobi.png', dpi=150, bbox_inches='tight')
plt.close()
